In this notebook, we showcase how to use the improve retrieval performance using per-layer compression.

In [ ]:
import numpy as np
import torch
from transformers import pipeline

from kvpress import (
    ExpectedAttentionPress,
    KnormPress,
    ObservedAttentionPress,
    RandomPress,
    SnapKVPress,
    StreamingLLMPress,
    PerLayerCompressionPress,
)

# Load the pipeline and data

In [ ]:
# Load pipeline

device = "cuda:0"
ckpt = "microsoft/Phi-3.5-mini-instruct"
attn_implementation = "flash_attention_2"
pipe = pipeline("kv-press-text-generation", model=ckpt, device=device, dtype="auto", model_kwargs={"attn_implementation":attn_implementation})

In [ ]:
import datasets 

df = datasets.load_dataset("simonjegou/ruler", "4096")["test"].to_pandas()
df = df.loc[df["task"] == "niah_single_3"].reset_index(drop=True)

# Use the pipeline with a press

In [ ]:
# Pick a press with a compression ratio, you can run the following cells with different presses
compression_ratio = 0.3
press = ExpectedAttentionPress(compression_ratio)

In [ ]:
# Run the pipeline on a single question
idx = 0
context = df.iloc[idx]["context"] 
question = df.iloc[idx]["question"] 
true_answer = df.iloc[idx]["answer"][0]

pred_answer = pipe(context, question=question, press=press)["answer"]

print(f"Question:   {question}")
print(f"Answer:     {true_answer}")
print(f"Prediction: {pred_answer}")
print(f"Correctly predicted: {true_answer in pred_answer}")

# Apply per-layer-compression with the same overall compression ratio

In [ ]:
# Each layer is compressed differently, some layers have higher compression ratios, other less.
# The mean compression ratio is the same as for the original press

PHI_35_COMPRESSION_RATIOS = [0.37, 0.3, 0.37, 0.37, 0.37, 0.37, 0.07, 0.37, 0.29, 0.37, 0.36,
                             0.13, 0.37, 0.0, 0.37, 0.37, 0.37, 0.36, 0.28, 0.0, 0.09, 0.37,
                             0.37, 0.37, 0.37, 0.37, 0.04, 0.37, 0.37, 0.37, 0.37, 0.37]
print(np.mean(PHI_35_COMPRESSION_RATIOS))

In [ ]:
press_per_layer = PerLayerCompressionPress(ExpectedAttentionPress(compression_ratio), PHI_35_COMPRESSION_RATIOS)

In [ ]:
pred_answer = pipe(context, question=question, press=press_per_layer)["answer"]

print(f"Question:   {question}")
print(f"Answer:     {true_answer}")
print(f"Prediction: {pred_answer}")
print(f"Correctly predicted: {true_answer in pred_answer}")